#  Analyse des Correspondances Multiples (ACM)
#  Profiling des Clients à Risque de Crédit Bancaire

---

##  Objectif du Projet

Ce notebook réalise une **Analyse des Correspondances Multiples (ACM)** sur un jeu de données de **crédit bancaire** afin de :

1. **Explorer** les profils des demandeurs de crédit à travers leurs caractéristiques socio-économiques
2. **Identifier** les associations entre les modalités des variables (ex : qui emprunte quoi ?)
3. **Segmenter** les clients en groupes homogènes pour mieux comprendre le risque
4. **Visualiser** les résultats sur des plans factoriels interactifs

---

##  Pourquoi l'ACM ?

> L'**ACM** (Multiple Correspondence Analysis) est l'équivalent de l'ACP pour les variables **qualitatives / catégorielles**.  
> Elle permet de résumer l'information contenue dans plusieurs variables catégorielles en un petit nombre d'axes factoriels,  
> tout en révélant les **associations cachées** entre les modalités.

| Méthode | Type de variables | Objectif |
|---------|------------------|----------|
| ACP     | Quantitatives     | Réduire la dimension, corrélations |
| AFC     | 2 variables qual. | Association entre 2 variables |
| **ACM** | **Multiple qual.**| **Profils, associations multiples** |

---

##  Description du Dataset

Le dataset contient des informations sur des **demandeurs de crédit bancaire** avec les variables :

| Variable | Description | Type |
|----------|-------------|------|
| `Marche` | Comportement du compte courant | Catégorielle |
| `Apport` | Niveau d'apport personnel | Catégorielle |
| `Impaye` | Historique d'impayés | Catégorielle |
| `Assurance` | Type d'assurance souscrite | Catégorielle |
| `Endettement` | Niveau d'endettement actuel | Catégorielle |
| `Famille` | Situation familiale | Catégorielle |
| `Enfants` | Nombre d'enfants | Catégorielle |
| `Logement` | Type de logement | Catégorielle |
| `Profession` | Catégorie professionnelle | Catégorielle |
| `Intitule` | Type de crédit demandé | Catégorielle |
| `Age` | Âge du demandeur | Quantitative |

---

##  Plan du Notebook

1. **Importation & Configuration** — packages, style, seed
2. **Chargement & Exploration** — aperçu, types, stats descriptives
3. **Nettoyage & Préparation** — valeurs manquantes, doublons, encodage
4. **Analyse Univariée** — distribution de chaque variable
5. **Analyse Bivariée** — tests du Chi², V de Cramér
6. **ACM Complète** — valeurs propres, cosinus², contributions
7. **Visualisations Factorielles** — plans factoriels annotés
8. **Clustering sur axes ACM** — K-Means pour segmenter les profils
9. **Interprétation & Insights** — synthèse métier
10. **Export des Résultats** — données pour l'app Streamlit

---
## Importation des Packages

On importe ici toutes les bibliothèques nécessaires :
- `pandas` / `numpy` : manipulation de données
- `matplotlib` / `seaborn` / `plotly` : visualisations
- `prince` : bibliothèque moderne pour l'ACM (MCA)
- `sklearn` : clustering K-Means
- `scipy` : tests statistiques (Chi²)

In [ ]:
# ============================================================
# BLOC 1 — IMPORTATION DES PACKAGES
# ============================================================

# --- Manipulation de données ---
import pandas as pd
import numpy as np

# --- Visualisations statiques ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# --- Visualisations interactives ---
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- ACM (Multiple Correspondence Analysis) ---
import prince  # bibliothèque moderne et bien maintenue pour l'ACM

# --- Clustering ---
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score

# --- Tests statistiques ---
from scipy.stats import chi2_contingency

# --- Gestion des warnings ---
import warnings
warnings.filterwarnings('ignore')

# --- Configuration globale des graphiques ---
plt.rcParams['figure.figsize'] = (12, 6)          # taille par défaut
plt.rcParams['axes.spines.top'] = False            # supprime le cadre haut
plt.rcParams['axes.spines.right'] = False          # supprime le cadre droit
sns.set_palette('Set2')                            # palette douce et accessible
RANDOM_SEED = 42                                   # reproductibilité des résultats

print(' Tous les packages sont importés avec succès !')
print(f'   • pandas  v{pd.__version__}')
print(f'   • numpy   v{np.__version__}')
print(f'   • prince  v{prince.__version__}')

---
## Cellule 2 : Génération du Dataset Simulé

> **Note :** En l'absence du fichier CSV original `credit_data_set.csv`, nous générons un dataset **réaliste et cohérent** de 1000 clients.  
> Les distributions et associations reproduisent les patterns typiques d'un portefeuille de crédit bancaire.

### Pourquoi simuler ?
- Reproductibilité totale sans dépendance externe
- Contrôle des associations (on peut injecter des corrélations réalistes)
- Dataset libre de droits pour le CV / portfolio

In [ ]:
# ============================================================
# BLOC 2 — GÉNÉRATION D'UN DATASET RÉALISTE (1000 clients)
# ============================================================

np.random.seed(RANDOM_SEED)
n = 1000  # nombre de clients dans notre portefeuille

# --- PROFILS DE BASE ---
# On crée 3 profils latents qui génèrent des associations réalistes :
#   Profil A : Bon payeur, CDI, stable    (~40% des clients)
#   Profil B : Risque modéré, CDD/Interim (~35%)
#   Profil C : Risque élevé, précaire     (~25%)

profils = np.random.choice(['A', 'B', 'C'], size=n, p=[0.40, 0.35, 0.25])

# --- VARIABLE : Marche (comportement compte courant) ---
# Profil A → compte bien géré, Profil C → souvent débiteur
marche_map = {
    'A': np.random.choice(['Positif', 'Positif_fort', 'Sans_compte'], p=[0.5, 0.4, 0.1]),
    'B': np.random.choice(['Positif', 'Nul', 'Débiteur_faible'], p=[0.4, 0.35, 0.25]),
    'C': np.random.choice(['Débiteur_faible', 'Débiteur_fort', 'Nul'], p=[0.4, 0.35, 0.25])
}
Marche = np.array([marche_map[p] if isinstance(marche_map[p], str) 
                   else marche_map[p] for p in profils])

# Reconstruction propre par profil
def gen_var(profil_array, choices_dict):
    result = []
    for p in profil_array:
        choices, probs = choices_dict[p]
        result.append(np.random.choice(choices, p=probs))
    return np.array(result)

Marche = gen_var(profils, {
    'A': (['Positif', 'Positif_fort', 'Sans_compte'], [0.50, 0.40, 0.10]),
    'B': (['Positif', 'Nul', 'Débiteur_faible'],      [0.40, 0.35, 0.25]),
    'C': (['Débiteur_faible', 'Débiteur_fort', 'Nul'],[0.40, 0.35, 0.25])
})

Apport = gen_var(profils, {
    'A': (['Elevé', 'Moyen', 'Faible'],   [0.45, 0.40, 0.15]),
    'B': (['Moyen', 'Faible', 'Elevé'],   [0.45, 0.40, 0.15]),
    'C': (['Faible', 'Très_faible', 'Moyen'], [0.50, 0.30, 0.20])
})

Impaye = gen_var(profils, {
    'A': (['Aucun', 'Ponctuel'],             [0.85, 0.15]),
    'B': (['Aucun', 'Ponctuel', 'Répété'],   [0.55, 0.30, 0.15]),
    'C': (['Répété', 'Ponctuel', 'Aucun'],   [0.50, 0.30, 0.20])
})

Assurance = gen_var(profils, {
    'A': (['Décès_invalidité', 'Complète', 'Aucune'], [0.50, 0.35, 0.15]),
    'B': (['Décès_invalidité', 'Aucune', 'Complète'], [0.45, 0.35, 0.20]),
    'C': (['Aucune', 'Décès_invalidité', 'Complète'], [0.55, 0.30, 0.15])
})

Endettement = gen_var(profils, {
    'A': (['Faible', 'Modéré', 'Elevé'],        [0.55, 0.35, 0.10]),
    'B': (['Modéré', 'Elevé', 'Faible'],        [0.50, 0.30, 0.20]),
    'C': (['Elevé', 'Très_élevé', 'Modéré'],    [0.45, 0.35, 0.20])
})

Famille = gen_var(profils, {
    'A': (['Marié', 'Célibataire', 'Divorcé'],  [0.55, 0.30, 0.15]),
    'B': (['Célibataire', 'Marié', 'Divorcé'],  [0.45, 0.35, 0.20]),
    'C': (['Célibataire', 'Divorcé', 'Marié'],  [0.45, 0.35, 0.20])
})

Enfants = gen_var(profils, {
    'A': (['0', '1-2', '3+'],  [0.25, 0.55, 0.20]),
    'B': (['0', '1-2', '3+'],  [0.40, 0.45, 0.15]),
    'C': (['0', '3+', '1-2'],  [0.35, 0.35, 0.30])
})

Logement = gen_var(profils, {
    'A': (['Propriétaire', 'Locataire', 'Hébergé'],  [0.55, 0.35, 0.10]),
    'B': (['Locataire', 'Propriétaire', 'Hébergé'],  [0.50, 0.30, 0.20]),
    'C': (['Locataire', 'Hébergé', 'Propriétaire'],  [0.50, 0.30, 0.20])
})

Profession = gen_var(profils, {
    'A': (['Cadre', 'Employé_qualifié', 'Fonctionnaire'],  [0.35, 0.40, 0.25]),
    'B': (['Employé_qualifié', 'Ouvrier', 'Cadre'],        [0.40, 0.35, 0.25]),
    'C': (['Ouvrier', 'Sans_emploi', 'Indépendant'],       [0.35, 0.35, 0.30])
})

Intitule = gen_var(profils, {
    'A': (['Immobilier', 'Voiture_neuve', 'Consommation'],  [0.45, 0.35, 0.20]),
    'B': (['Voiture_neuve', 'Consommation', 'Immobilier'],  [0.40, 0.35, 0.25]),
    'C': (['Consommation', 'Voiture_occasion', 'Voiture_neuve'], [0.45, 0.30, 0.25])
})

# --- VARIABLE : Age (quantitative → sera catégorisée) ---
Age_base = {'A': (42, 10), 'B': (35, 8), 'C': (30, 9)}
Age = np.array([int(np.clip(np.random.normal(*Age_base[p]), 20, 75)) for p in profils])

# --- VARIABLE CIBLE : Risque (pour valider nos interprétations) ---
Risque = np.where(profils == 'A', 'Bon', np.where(profils == 'B', 'Moyen', 'Mauvais'))

# --- CONSTRUCTION DU DATAFRAME FINAL ---
credit_data = pd.DataFrame({
    'Marche': Marche, 'Apport': Apport, 'Impaye': Impaye,
    'Assurance': Assurance, 'Endettement': Endettement, 'Famille': Famille,
    'Enfants': Enfants, 'Logement': Logement, 'Profession': Profession,
    'Intitule': Intitule, 'Age': Age, 'Profil_latent': profils, 'Risque': Risque
})

# --- Catégorisation de l'âge ---
credit_data['Age_groupe'] = pd.cut(
    credit_data['Age'],
    bins=[19, 29, 39, 49, 59, 100],
    labels=['20-29', '30-39', '40-49', '50-59', '60+']
)

print(f' Dataset généré : {credit_data.shape[0]} clients × {credit_data.shape[1]} variables')
print(f'\n Répartition des profils latents :')
print(credit_data['Profil_latent'].value_counts().rename({'A':'Bon payeur','B':'Risque modéré','C':'Risque élevé'}))
credit_data.head()

---
##  Cellule 3 : Exploration et Qualité des Données

Avant toute analyse, on doit comprendre la **structure** du jeu de données :
- Combien de lignes / colonnes ?
- Quels types de données ?
- Y a-t-il des valeurs manquantes ? Des doublons ?

C'est l'étape de **diagnostic** indispensable.

In [ ]:
# ============================================================
# BLOC 3 — EXPLORATION ET QUALITÉ DES DONNÉES
# ============================================================

print('=' * 60)
print('RAPPORT DE QUALITÉ DES DONNÉES')
print('=' * 60)

# 1) Dimensions
print(f'\n Dimensions : {credit_data.shape[0]} lignes × {credit_data.shape[1]} colonnes')

# 2) Types de colonnes
print(f'\n Types de données :')
for col in credit_data.columns:
    n_unique = credit_data[col].nunique()
    print(f'   {col:<20} {str(credit_data[col].dtype):<12} → {n_unique} modalités/valeurs')

# 3) Valeurs manquantes
missing = credit_data.isnull().sum()
missing_pct = (missing / len(credit_data) * 100).round(2)
print(f'\n  Valeurs manquantes :')
if missing.sum() == 0:
    print('    Aucune valeur manquante — dataset propre !')
else:
    print(pd.DataFrame({'Nb manquants': missing[missing>0], '% manquants': missing_pct[missing>0]}))

# 4) Doublons
n_dup = credit_data.duplicated().sum()
print(f'\nDoublons : {n_dup} lignes dupliquées', '✅' if n_dup == 0 else '⚠️  À traiter !')

# 5) Statistiques descriptives des variables qualitatives
print(f'\n Statistiques descriptives (variables catégorielles) :')
cat_cols = ['Marche', 'Apport', 'Impaye', 'Assurance', 'Endettement',
            'Famille', 'Enfants', 'Logement', 'Profession', 'Intitule']
credit_data[cat_cols].describe()

---
##  Cellule 4 : Analyse Univariée — Distribution des Variables

On analyse chaque variable **séparément** pour comprendre la composition du portefeuille.

**Question métier :** Quels sont les profils démographiques et comportementaux dominants dans notre portefeuille ?

In [ ]:
# ============================================================
# BLOC 4 — ANALYSE UNIVARIÉE
# ============================================================

# On crée une grille de graphiques : 5 lignes × 2 colonnes
fig, axes = plt.subplots(5, 2, figsize=(16, 25))
axes = axes.flatten()  # aplati pour itérer facilement

# Palette de couleurs par variable pour différencier visuellement
palettes = ['Blues_d', 'Greens_d', 'Reds_d', 'Purples_d', 'Oranges_d',
            'PuBu', 'YlGn', 'OrRd', 'BuPu', 'GnBu']

for i, col in enumerate(cat_cols):
    # Calcul des proportions
    freq = credit_data[col].value_counts(normalize=True).sort_values(ascending=False) * 100
    
    # Graphique en barres horizontales (plus lisible quand les labels sont longs)
    bars = axes[i].barh(
        freq.index, freq.values,
        color=sns.color_palette(palettes[i], len(freq))
    )
    
    # Ajout des pourcentages sur les barres
    for bar, val in zip(bars, freq.values):
        axes[i].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
    
    axes[i].set_title(f'Distribution de {col}', fontsize=13, fontweight='bold', pad=10)
    axes[i].set_xlabel('Proportion (%)', fontsize=10)
    axes[i].set_xlim(0, freq.max() * 1.15)  # marge pour les labels
    axes[i].invert_yaxis()  # modalité la plus fréquente en haut

plt.suptitle('Analyse Univariée — Distribution des Variables Catégorielles\n',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('univarie_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Tableau récapitulatif des modalités dominantes ---
print('\n📋 Modalités dominantes par variable :')
print('-' * 55)
for col in cat_cols:
    top = credit_data[col].value_counts(normalize=True).iloc[0]
    top_label = credit_data[col].value_counts().index[0]
    print(f'{col:<20} → {top_label:<25} ({top*100:.1f}%)')

---
## 🔗 Cellule 5 : Analyse Bivariée — Tests du Chi² et V de Cramér

L'ACM analyse les **associations** entre variables. Avant d'appliquer l'ACM, on mesure ces associations avec :

### Test du Chi² (χ²)
> Teste l'**indépendance** entre deux variables catégorielles.  
> H₀ : les deux variables sont indépendantes.  
> Si p-value < 0.05 → on rejette H₀ → **les variables sont liées**.

### V de Cramér
> Mesure l'**intensité** de l'association (0 = aucune, 1 = parfaite).  
> Interprétation standard :
> - 0.0 – 0.1 : très faible
> - 0.1 – 0.3 : modérée  
> - 0.3 – 0.5 : forte
> - > 0.5     : très forte

In [ ]:
# ============================================================
# BLOC 5 — MATRICE DES V DE CRAMÉR
# ============================================================

def cramers_v(x, y):
    """
    Calcule le V de Cramér entre deux séries catégorielles.
    
    Paramètres :
    - x, y : pd.Series avec les modalités
    
    Retourne :
    - v : float [0, 1] — force de l'association
    - p : float — p-value du test Chi²
    """
    # Tableau de contingence
    ct = pd.crosstab(x, y)
    chi2, p, dof, expected = chi2_contingency(ct)
    
    n = ct.sum().sum()
    r, k = ct.shape
    
    # Formule du V de Cramér (corrigé pour le biais)
    phi2 = max(0, chi2/n - (k-1)*(r-1)/(n-1))
    r_corr = r - (r-1)**2/(n-1)
    k_corr = k - (k-1)**2/(n-1)
    v = np.sqrt(phi2 / min(r_corr-1, k_corr-1)) if min(r_corr-1, k_corr-1) > 0 else 0
    
    return round(v, 3), round(p, 4)

# Calcul de toutes les paires de variables
cramer_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
pvalue_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)

for col1 in cat_cols:
    for col2 in cat_cols:
        if col1 == col2:
            cramer_matrix.loc[col1, col2] = 1.0
            pvalue_matrix.loc[col1, col2] = 0.0
        else:
            v, p = cramers_v(credit_data[col1], credit_data[col2])
            cramer_matrix.loc[col1, col2] = v
            pvalue_matrix.loc[col1, col2] = p

cramer_matrix = cramer_matrix.astype(float)

# --- Heatmap du V de Cramér ---
fig, ax = plt.subplots(figsize=(13, 10))

mask = np.zeros_like(cramer_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # on masque le triangle supérieur (redondant)

sns.heatmap(
    cramer_matrix,
    annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    ax=ax, square=True,
    cbar_kws={'label': "V de Cramér (force de l'association)"}
)

ax.set_title('Matrice des V de Cramér\n(vert = association forte, rouge = association faible)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('cramer_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Top associations ---
print('\n🔝 Top 10 des associations les plus fortes :')
pairs = []
for i, c1 in enumerate(cat_cols):
    for j, c2 in enumerate(cat_cols):
        if i < j:
            pairs.append({'Variable 1': c1, 'Variable 2': c2,
                          'V de Cramér': cramer_matrix.loc[c1, c2],
                          'p-value': pvalue_matrix.loc[c1, c2]})

top_pairs = pd.DataFrame(pairs).sort_values('V de Cramér', ascending=False).head(10)
top_pairs['Significatif'] = top_pairs['p-value'].apply(lambda p: '✅' if p < 0.05 else '❌')
print(top_pairs.to_string(index=False))

---
## Cellule 6 : Préparation pour l'ACM

L'ACM nécessite :
1. Un tableau de **variables qualitatives uniquement**
2. Les modalités encodées de façon lisible (pas de codes numériques)
3. Le choix du **nombre d'axes** à conserver

### Variables retenues pour l'ACM
On sélectionne les variables ayant les associations les plus fortes (V de Cramér) pour maximiser l'information extraite.

In [ ]:
# ============================================================
# BLOC 6 — PRÉPARATION POUR L'ACM
# ============================================================

# --- Sélection des variables pour l'ACM ---
# On garde les variables les plus informatives pour le risque crédit
acm_vars = ['Marche', 'Assurance', 'Intitule', 'Impaye', 'Profession', 'Endettement']

# Tableau de données pour l'ACM (uniquement les colonnes sélectionnées)
X_acm = credit_data[acm_vars].copy()

print(' Données pour l\'ACM :')
print(f'   • {X_acm.shape[0]} individus (clients)')
print(f'   • {X_acm.shape[1]} variables actives : {acm_vars}')
print(f'   • Nombre total de modalités : {sum(X_acm[v].nunique() for v in acm_vars)}')

print('\n Modalités par variable :')
for v in acm_vars:
    mods = sorted(X_acm[v].unique())
    print(f'   {v:<20} ({len(mods)} modalités) : {mods}')

# Vérification : aucune valeur manquante dans le sous-ensemble
assert X_acm.isnull().sum().sum() == 0, "⚠️ Des valeurs manquantes existent !"
print('\n Aucune valeur manquante dans le sous-ensemble ACM')
print(' Données prêtes pour l\'ACM')

---
##  Cellule 7 : Application de l'ACM avec `prince`

On utilise la bibliothèque **`prince`** qui implémente l'ACM de façon moderne et efficace.

### Paramètres clés
- `n_components` : nombre d'axes factoriels à calculer (on en calcule 10, on en retient moins)
- `n_iter` : nombre d'itérations pour l'algorithme SVD
- `random_state` : graine aléatoire pour la reproductibilité

### Ce que l'ACM calcule
1. **Valeurs propres** (eigenvalues) → part de variance expliquée par chaque axe
2. **Coordonnées des individus** → position de chaque client dans l'espace factoriel
3. **Coordonnées des modalités** → position de chaque catégorie dans l'espace factoriel
4. **Contributions** → qui/quoi contribue le plus à construire chaque axe ?
5. **Cosinus²** → qualité de représentation sur chaque axe

In [ ]:
# ============================================================
# BLOC 7 — APPLICATION DE L'ACM
# ============================================================

# Instanciation de l'ACM
# n_components=10 : on calcule les 10 premiers axes
#                   (on en choisira 2 ou 3 selon les valeurs propres)
mca = prince.MCA(
    n_components=10,
    n_iter=10,
    copy=True,
    check_input=True,
    random_state=RANDOM_SEED,
    engine='sklearn'  # moteur de calcul SVD
)

# Ajustement sur nos données (fit)
# À l'issue, mca contient toutes les statistiques calculées
mca.fit(X_acm)

print(' ACM ajustée avec succès !')
print(f'   Algorithme utilisé : Décomposition en Valeurs Singulières (SVD)')
print(f'   Nombre d\'axes calculés : {mca.n_components}')

# --- Résumé des valeurs propres ---
eigenvalues = mca.eigenvalues_
explained_var = mca.percentage_of_variance_ if hasattr(mca, 'percentage_of_variance_') else None

# Calcul manuel de la variance expliquée
total_inertia = sum(eigenvalues)
pct_var = [round(e / total_inertia * 100, 2) for e in eigenvalues]
cumul_var = [round(sum(pct_var[:i+1]), 2) for i in range(len(pct_var))]

print(f'\n Tableau des valeurs propres :')
print(f'   Inertie totale : {total_inertia:.4f}')
print('\n   Axe | Valeur propre | % Variance | % Cumulé')
print('   ' + '-' * 48)
for i, (e, p, c) in enumerate(zip(eigenvalues, pct_var, cumul_var), 1):
    flag = ' ← retenir' if c <= 75 and i <= 5 else ''
    print(f'   {i:>3} | {e:>13.4f} | {p:>10.2f}% | {c:>8.2f}%{flag}')

---
##  Cellule 8 : Éboulis des Valeurs Propres

Le **graphique des valeurs propres** (scree plot / éboulis) aide à choisir le nombre d'axes à retenir.

### Règles de choix
1. **Règle du coude** : on s'arrête là où la courbe « se casse »
2. **Règle du seuil** : on retient les axes qui expliquent ensemble ≥ 70-80% de la variance
3. **Règle de Kaiser** : en ACM, on retient les axes dont λ > 1/K (K = nombre de variables)

In [ ]:
# ============================================================
# BLOC 8 — ÉBOULIS DES VALEURS PROPRES
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

n_axes_display = min(10, len(eigenvalues))  # on affiche au max 10 axes
axes_labels = [f'Axe {i+1}' for i in range(n_axes_display)]

# --- Graphique 1 : Valeurs propres brutes ---
ax1 = axes[0]
ax1.bar(axes_labels, eigenvalues[:n_axes_display], color='steelblue', edgecolor='white')
ax1.plot(axes_labels, eigenvalues[:n_axes_display], 'o-', color='darkblue', markersize=8)
# Ligne de référence Kaiser : 1/K (K = nombre de variables)
kaiser_threshold = 1 / len(acm_vars)
ax1.axhline(y=kaiser_threshold, color='red', linestyle='--', label=f'Seuil Kaiser (1/K={kaiser_threshold:.3f})')
ax1.set_title('Valeurs Propres (Eigenvalues)', fontweight='bold')
ax1.set_ylabel('Valeur propre')
ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=45)

# --- Graphique 2 : % Variance expliquée ---
ax2 = axes[1]
colors = ['#2196F3' if p > 5 else '#B0BEC5' for p in pct_var[:n_axes_display]]  # bleu si important
ax2.bar(axes_labels, pct_var[:n_axes_display], color=colors, edgecolor='white')
for i, p in enumerate(pct_var[:n_axes_display]):
    ax2.text(i, p + 0.2, f'{p:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax2.set_title('Variance Expliquée par Axe (%)', fontweight='bold')
ax2.set_ylabel('% Variance')
ax2.tick_params(axis='x', rotation=45)

# --- Graphique 3 : Variance cumulée ---
ax3 = axes[2]
ax3.plot(axes_labels, cumul_var[:n_axes_display], 'o-', color='#4CAF50', linewidth=2.5, markersize=8)
ax3.fill_between(range(n_axes_display), cumul_var[:n_axes_display], alpha=0.15, color='#4CAF50')
# Seuil 80% 
ax3.axhline(y=80, color='orange', linestyle='--', label='Seuil 80%')
ax3.axhline(y=70, color='red', linestyle=':', label='Seuil 70%')
for i, c in enumerate(cumul_var[:n_axes_display]):
    ax3.text(i, c + 1, f'{c:.0f}%', ha='center', fontsize=8)
ax3.set_title('Variance Cumulée (%)', fontweight='bold')
ax3.set_ylabel('% Variance cumulée')
ax3.set_ylim(0, 110)
ax3.legend(fontsize=8)
ax3.tick_params(axis='x', rotation=45)

plt.suptitle('Éboulis des Valeurs Propres — Choix du Nombre d\'Axes',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eboulis_valeurs_propres.png', dpi=150, bbox_inches='tight')
plt.show()

# Recommandation automatique
axes_retenus = next(i+1 for i, c in enumerate(cumul_var) if c >= 70)
print(f'\n Recommandation : retenir les {axes_retenus} premiers axes')
print(f'   → Ils expliquent {cumul_var[axes_retenus-1]:.1f}% de l\'inertie totale')
print(f'   → On travaillera principalement avec les axes 1 et 2 (plan principal)')

---
##  Cellule 9 : Coordonnées, Contributions et Cosinus²

### Trois indicateurs essentiels de l'ACM

| Indicateur | Ce qu'il mesure | Bonne valeur |
|------------|----------------|-------------|
| **Coordonnées** | Position sur l'axe factoriel | — (valeur signée) |
| **Cosinus²** (cos²) | **Qualité** de représentation | > 0.5 = bien représenté |
| **Contribution** (CTR) | Poids dans la **construction** de l'axe | > 1/nb_modalités = important |

>  **Règle importante** : une modalité mal représentée (faible cos²) ne doit pas être interprétée !

In [ ]:
# ============================================================
# BLOC 9 — COORDONNÉES, CONTRIBUTIONS ET COS² DES MODALITÉS
# ============================================================

# --- Coordonnées des modalités (colonnes) dans l'espace factoriel ---
col_coords = mca.column_coordinates(X_acm)
# Rename axes pour lisibilité
col_coords.columns = [f'Axe_{i+1}' for i in range(col_coords.shape[1])]

# --- Coordonnées des individus (lignes) ---
row_coords = mca.row_coordinates(X_acm)
row_coords.columns = [f'Axe_{i+1}' for i in range(row_coords.shape[1])]

print(' Coordonnées des modalités (5 premières lignes) :')
print(col_coords.head())

# --- Contributions des modalités ---
# Contribution = part de la modalité dans la construction de l'axe
# = (effectif_modalité * coord²) / valeur_propre
n_total = len(X_acm)
contrib_rows = []
for mod in col_coords.index:
    # Retrouver la variable correspondante
    var_name = mod.split('_')[0] if '_' in mod else mod
    # Effectif de la modalité
    n_mod = sum(X_acm[v].astype(str).eq(str(mod).split('_', 1)[-1]).sum() 
                for v in acm_vars)
    contrib_rows.append(mod)

# Calcul des cos² : cos²_j_a = coord²_j_a / distance²_j
# distance² = somme des coord² sur tous les axes
dist2 = (col_coords ** 2).sum(axis=1)
cos2_cols = col_coords.copy()
for col in cos2_cols.columns:
    cos2_cols[col] = cos2_cols[col] ** 2 / dist2

# Renommage pour clarté
cos2_cols.columns = [f'cos2_Axe_{i+1}' for i in range(cos2_cols.shape[1])]

print('\n Qualité de représentation (cos²) des modalités — Axes 1 et 2 :')
cos2_display = cos2_cols[['cos2_Axe_1', 'cos2_Axe_2']].copy()
cos2_display['cos2_total'] = cos2_display.sum(axis=1)
cos2_display = cos2_display.sort_values('cos2_total', ascending=False)
cos2_display['Qualité'] = cos2_display['cos2_total'].apply(
    lambda x: '🟢 Bien repr.' if x > 0.5 else ('🟡 Moyen' if x > 0.3 else '🔴 Mal repr.')
)
print(cos2_display.round(3).to_string())

---
##  Cellule 10 : Plan Factoriel — Carte des Modalités (Axes 1 & 2)

Le **plan factoriel** est la visualisation centrale de l'ACM.  
Chaque point représente une **modalité** d'une variable.

### Règles d'interprétation
1. **Proximité** : deux modalités proches → elles sont souvent observées ensemble
2. **Distance à l'origine** : plus une modalité est loin → plus elle est caractéristique
3. **Sens des axes** : les axes opposent des profils distincts (ex: axe 1 = risque faible ↔ risque fort)
4. **Groupes visuels** : les nuages de points révèlent des profils-types

In [ ]:
# ============================================================
# BLOC 10 — PLAN FACTORIEL ANNOTÉ (Axes 1 & 2)
# ============================================================

fig, ax = plt.subplots(figsize=(14, 10))

# Couleurs par variable d'origine
var_colors = {
    'Marche': '#E74C3C',        # rouge
    'Assurance': '#3498DB',     # bleu
    'Intitule': '#2ECC71',      # vert
    'Impaye': '#F39C12',        # orange
    'Profession': '#9B59B6',    # violet
    'Endettement': '#1ABC9C'    # turquoise
}

# Marqueurs par variable
var_markers = {'Marche': 'o', 'Assurance': 's', 'Intitule': '^', 
               'Impaye': 'D', 'Profession': 'P', 'Endettement': '*'}

# Tracer les modalités
legend_handles = []
for mod in col_coords.index:
    # Identifier la variable d'appartenance
    var = next((v for v in acm_vars if any(
        str(val) in str(mod) for val in credit_data[v].unique()
    )), None)
    if var is None:
        var = acm_vars[0]  # fallback
    
    x = col_coords.loc[mod, 'Axe_1']
    y = col_coords.loc[mod, 'Axe_2']
    
    color = var_colors.get(var, 'gray')
    marker = var_markers.get(var, 'o')
    
    ax.scatter(x, y, c=color, marker=marker, s=150, zorder=5, alpha=0.9,
               edgecolors='white', linewidth=0.5)
    
    # Label de la modalité
    label = str(mod).split('_', 1)[-1] if '_' in str(mod) else str(mod)
    ax.annotate(label, (x, y),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6, edgecolor='none'))

# Légende par variable
for var, color in var_colors.items():
    patch = mpatches.Patch(color=color, label=var)
    legend_handles.append(patch)

# Lignes d'axes passant par l'origine
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

# Labels et titre
ax.set_xlabel(f'Axe 1 ({pct_var[0]:.1f}% de variance)', fontsize=12, fontweight='bold')
ax.set_ylabel(f'Axe 2 ({pct_var[1]:.1f}% de variance)', fontsize=12, fontweight='bold')
ax.set_title('Plan Factoriel ACM — Carte des Modalités\n(Axes 1 & 2)',
             fontsize=14, fontweight='bold', pad=15)

ax.legend(handles=legend_handles, title='Variables', loc='upper right',
          framealpha=0.9, fontsize=9)

# Grille légère
ax.grid(True, alpha=0.2, linestyle=':')

plt.tight_layout()
plt.savefig('plan_factoriel_modalites.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Lecture du graphique :')
print('   • Modalités proches = souvent observées ensemble')
print('   • Distance à l\'origine = caractère distinctif')
print('   • Axe 1 : oppose les profils de risque (faible ↔ élevé)')
print('   • Axe 2 : différencie les types de crédit et de profession')

---
## 👥 Cellule 11 : Plan des Individus coloré par Risque

On projette les **individus (clients)** dans le plan factoriel et on les colore par leur **niveau de risque**.  
Cela permet de valider visuellement que l'ACM capture bien la dimension risque.

In [ ]:
# ============================================================
# BLOC 11 — PLAN DES INDIVIDUS COLORÉ PAR RISQUE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Couleurs pour le risque
risque_colors = {'Bon': '#2ECC71', 'Moyen': '#F39C12', 'Mauvais': '#E74C3C'}
risque_labels = credit_data['Risque'].values

# --- Graphique 1 : Individus colorés par risque ---
for risque_val, color in risque_colors.items():
    mask = risque_labels == risque_val
    axes[0].scatter(
        row_coords.loc[mask, 'Axe_1'],
        row_coords.loc[mask, 'Axe_2'],
        c=color, label=f'Risque {risque_val}',
        alpha=0.4, s=20, edgecolors='none'
    )
    # Centroïde du groupe
    cx = row_coords.loc[mask, 'Axe_1'].mean()
    cy = row_coords.loc[mask, 'Axe_2'].mean()
    axes[0].scatter(cx, cy, c=color, s=300, marker='*', 
                    edgecolors='black', linewidth=1.5, zorder=10)
    axes[0].annotate(f'Centroïde\n{risque_val}', (cx, cy),
                     xytext=(8, 8), textcoords='offset points',
                     fontsize=9, fontweight='bold', color=color)

axes[0].axhline(y=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_xlabel(f'Axe 1 ({pct_var[0]:.1f}%)', fontsize=11, fontweight='bold')
axes[0].set_ylabel(f'Axe 2 ({pct_var[1]:.1f}%)', fontsize=11, fontweight='bold')
axes[0].set_title('Individus colorés par Niveau de Risque', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.2)

# --- Graphique 2 : Distribution sur l'Axe 1 par risque (boxplot) ---
# L'axe 1 est généralement l'axe qui capture le mieux le risque
df_plot = pd.DataFrame({
    'Axe_1': row_coords['Axe_1'].values,
    'Risque': risque_labels
})

order = ['Bon', 'Moyen', 'Mauvais']
sns.violinplot(data=df_plot, x='Risque', y='Axe_1', order=order,
               palette=risque_colors, ax=axes[1], inner='quartile')
axes[1].set_title('Distribution sur l\'Axe 1 par Niveau de Risque\n(Violin Plot)', 
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('Niveau de Risque', fontsize=11)
axes[1].set_ylabel(f'Coordonnée sur Axe 1', fontsize=11)
axes[1].axhline(y=0, color='gray', linewidth=1, linestyle='--', alpha=0.5)
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle('Projection des Clients dans l\'Espace Factoriel ACM',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plan_individus_risque.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques des coordonnées par groupe
print('\n Coordonnées moyennes par niveau de risque (Axe 1) :')
print(df_plot.groupby('Risque')['Axe_1'].agg(['mean', 'std']).round(3))

---
## 🔵 Cellule 12 : Clustering K-Means sur les Axes ACM

On applique un **K-Means** sur les coordonnées factorielles pour :
1. Segmenter les clients en groupes homogènes
2. Donner des noms métier à chaque segment
3. Alimenter l'application Streamlit avec des profils actionnables

### Pourquoi K-Means sur les axes ACM ?
- Les axes ACM éliminent le bruit et les corrélations entre variables
- On clustérise sur une représentation compacte et débruitée
- Les clusters sont plus stables et interprétables

In [ ]:
# ============================================================
# BLOC 12 — CLUSTERING K-MEANS SUR LES AXES ACM
# ============================================================

# --- Choix du nombre de clusters (méthode du coude + silhouette) ---
# On utilise les 3 premiers axes factoriels pour le clustering
n_axes_cluster = min(3, row_coords.shape[1])
X_cluster = row_coords.iloc[:, :n_axes_cluster].values

# Test de K = 2 à 8
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, labels))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters K', fontsize=11)
axes[0].set_ylabel('Inertie intra-cluster', fontsize=11)
axes[0].set_title('Méthode du Coude (Elbow Method)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

best_k = K_range[silhouettes.index(max(silhouettes))]
axes[1].plot(K_range, silhouettes, 'o-', color='#E74C3C', linewidth=2, markersize=8)
axes[1].axvline(x=best_k, color='gray', linestyle='--', label=f'Meilleur K={best_k}')
axes[1].set_xlabel('Nombre de clusters K', fontsize=11)
axes[1].set_ylabel('Score de Silhouette', fontsize=11)
axes[1].set_title('Score de Silhouette par K', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Choix du Nombre Optimal de Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Application du K-Means avec K optimal (ou K=3 pour cohérence métier) ---
K_final = 3  # 3 segments = Bon / Moyen / Risqué → cohérent avec la logique crédit
kmeans_final = KMeans(n_clusters=K_final, random_state=RANDOM_SEED, n_init=20)
credit_data['Cluster'] = kmeans_final.fit_predict(X_cluster)

print(f'\n K-Means ajusté avec K={K_final} clusters')
print(f'   Score de silhouette : {silhouette_score(X_cluster, credit_data["Cluster"]):.3f}')
print(f'\n Taille des clusters :')
print(credit_data['Cluster'].value_counts().sort_index())

---
##  Cellule 13 : Profiling et Nommage des Clusters

On analyse la composition de chaque cluster pour lui donner un **nom métier** parlant.

In [ ]:
# ============================================================
# BLOC 13 — PROFILING DES CLUSTERS
# ============================================================

# Ajout des coordonnées ACM au dataframe
credit_data['ACM_Axe1'] = row_coords['Axe_1'].values
credit_data['ACM_Axe2'] = row_coords['Axe_2'].values

# Nommage des clusters (à ajuster selon les résultats réels)
cluster_names = {}
for c in range(K_final):
    sub = credit_data[credit_data['Cluster'] == c]
    risque_mode = sub['Risque'].value_counts().index[0]
    cluster_names[c] = f'Cluster {c} — Risque {risque_mode}'

credit_data['Segment'] = credit_data['Cluster'].map(cluster_names)

# Profil de chaque cluster
print('=' * 70)
print('PROFIL DÉTAILLÉ DES CLUSTERS')
print('=' * 70)

for c in range(K_final):
    sub = credit_data[credit_data['Cluster'] == c]
    print(f'\n🔵 {cluster_names[c]} — {len(sub)} clients ({len(sub)/len(credit_data)*100:.1f}%)')
    print('   ' + '-' * 60)
    for var in acm_vars:
        top_mod = sub[var].value_counts(normalize=True).iloc[0]
        top_name = sub[var].value_counts().index[0]
        print(f'   {var:<20} → {top_name:<25} ({top_mod*100:.0f}%)')
    print(f'   {"Âge moyen":<20} → {sub["Age"].mean():.1f} ans')
    print(f'   {"Risque (dominant)":<20} → {sub["Risque"].value_counts().index[0]}')

# Visualisation des clusters sur le plan factoriel
fig, ax = plt.subplots(figsize=(12, 8))

cluster_colors = ['#3498DB', '#E74C3C', '#2ECC71']
for c in range(K_final):
    mask = credit_data['Cluster'] == c
    ax.scatter(
        credit_data.loc[mask, 'ACM_Axe1'],
        credit_data.loc[mask, 'ACM_Axe2'],
        c=cluster_colors[c], alpha=0.4, s=20,
        label=cluster_names[c]
    )
    # Centroïde
    cx = credit_data.loc[mask, 'ACM_Axe1'].mean()
    cy = credit_data.loc[mask, 'ACM_Axe2'].mean()
    ax.scatter(cx, cy, c=cluster_colors[c], s=400, marker='*', 
               edgecolors='black', linewidth=2, zorder=10)

# Superposition des modalités
for mod in col_coords.index:
    x = col_coords.loc[mod, 'Axe_1']
    y = col_coords.loc[mod, 'Axe_2']
    ax.scatter(x, y, c='black', s=60, marker='D', zorder=8, alpha=0.7)
    label = str(mod).split('_', 1)[-1] if '_' in str(mod) else str(mod)
    ax.annotate(label, (x, y), xytext=(4, 4), textcoords='offset points',
                fontsize=7, color='black', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='lightyellow', alpha=0.7))

ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel(f'Axe 1 ({pct_var[0]:.1f}%)', fontsize=12, fontweight='bold')
ax.set_ylabel(f'Axe 2 ({pct_var[1]:.1f}%)', fontsize=12, fontweight='bold')
ax.set_title('Clusters K-Means sur le Plan Factoriel ACM\n(Individus + Modalités)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('clustering_acm.png', dpi=150, bbox_inches='tight')
plt.show()

---
##  Cellule 14 : Visualisation Interactive avec Plotly

On crée une version **interactive** du plan factoriel — idéale pour l'exploration et le CV !

In [ ]:
# ============================================================
# BLOC 14 — PLAN FACTORIEL INTERACTIF (PLOTLY)
# ============================================================

# --- Préparation des données pour Plotly ---
# Individus
df_individuals = credit_data[['ACM_Axe1', 'ACM_Axe2', 'Risque', 'Segment',
                               'Profession', 'Logement', 'Age']].copy()
df_individuals['Type'] = 'Individu'

# Modalités
col_data = []
for mod in col_coords.index:
    x = col_coords.loc[mod, 'Axe_1']
    y = col_coords.loc[mod, 'Axe_2']
    var = next((v for v in acm_vars if any(str(val) in str(mod) 
                for val in credit_data[v].unique())), 'Autre')
    label = str(mod).split('_', 1)[-1] if '_' in str(mod) else str(mod)
    col_data.append({'x': x, 'y': y, 'variable': var, 'modalite': label})

df_modalities = pd.DataFrame(col_data)

# --- Graphique interactif ---
fig = go.Figure()

# Couche 1 : Individus (semi-transparents)
for risque_val, color in [('Bon', '#2ECC71'), ('Moyen', '#F39C12'), ('Mauvais', '#E74C3C')]:
    mask = df_individuals['Risque'] == risque_val
    fig.add_trace(go.Scatter(
        x=df_individuals.loc[mask, 'ACM_Axe1'],
        y=df_individuals.loc[mask, 'ACM_Axe2'],
        mode='markers',
        name=f'Client Risque {risque_val}',
        marker=dict(color=color, size=5, opacity=0.4),
        hovertemplate=f'Risque: {risque_val}<br>Axe1: %{{x:.3f}}<br>Axe2: %{{y:.3f}}<extra></extra>'
    ))

# Couche 2 : Modalités (grands symboles)
for var in acm_vars:
    df_v = df_modalities[df_modalities['variable'] == var]
    if len(df_v) > 0:
        fig.add_trace(go.Scatter(
            x=df_v['x'], y=df_v['y'],
            mode='markers+text',
            name=var,
            text=df_v['modalite'],
            textposition='top center',
            marker=dict(size=14, symbol='diamond', line=dict(width=2, color='white')),
            hovertemplate='<b>%{text}</b><br>Variable: ' + var + 
                         '<br>Axe1: %{x:.3f}<br>Axe2: %{y:.3f}<extra></extra>'
        ))

# Mise en forme
fig.update_layout(
    title=dict(
        text='Plan Factoriel ACM — Analyse du Risque Crédit<br><sup>Cliquez sur la légende pour afficher/masquer les couches</sup>',
        font=dict(size=16)
    ),
    xaxis=dict(title=f'Axe 1 ({pct_var[0]:.1f}% de variance)', zeroline=True, zerolinewidth=1.5, zerolinecolor='gray'),
    yaxis=dict(title=f'Axe 2 ({pct_var[1]:.1f}% de variance)', zeroline=True, zerolinewidth=1.5, zerolinecolor='gray'),
    height=650,
    legend=dict(itemsizing='constant', groupclick='toggleitem'),
    plot_bgcolor='rgba(250,250,250,0.8)',
    hovermode='closest'
)

fig.show()
fig.write_html('plan_factoriel_interactif.html')
print('\n✅ Graphique interactif sauvegardé : plan_factoriel_interactif.html')

---
##  Cellule 15 : Export des Données pour Streamlit

On exporte tous les résultats nécessaires à l'application Streamlit.

In [ ]:
# ============================================================
# BLOC 15 — EXPORT DES RÉSULTATS POUR STREAMLIT
# ============================================================

import pickle

# 1) Dataset enrichi (avec clusters et coordonnées ACM)
credit_data.to_csv('credit_data_enriched.csv', index=False)
print(' credit_data_enriched.csv sauvegardé')

# 2) Coordonnées des modalités
col_coords.to_csv('mca_col_coordinates.csv')
print(' mca_col_coordinates.csv sauvegardé')

# 3) Coordonnées des individus
row_coords.to_csv('mca_row_coordinates.csv')
print(' mca_row_coordinates.csv sauvegardé')

# 4) Modèle ACM et K-Means sérialisés
with open('mca_model.pkl', 'wb') as f:
    pickle.dump(mca, f)
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans_final, f)
print(' mca_model.pkl sauvegardé')
print(' kmeans_model.pkl sauvegardé')

# 5) Métadonnées pour l'app
meta = {
    'acm_vars': acm_vars,
    'cat_cols': cat_cols,
    'pct_var': pct_var,
    'cumul_var': cumul_var,
    'eigenvalues': list(eigenvalues),
    'n_axes': axes_retenus,
    'k_clusters': K_final,
    'cluster_names': cluster_names,
    'cramer_matrix': cramer_matrix.to_dict()
}
with open('mca_metadata.pkl', 'wb') as f:
    pickle.dump(meta, f)
print(' mca_metadata.pkl sauvegardé')

print('\n Fichiers générés pour l\'app Streamlit :')
print('   credit_data_enriched.csv   → Dataset complet avec clusters')
print('   mca_col_coordinates.csv    → Coordonnées des modalités')
print('   mca_row_coordinates.csv    → Coordonnées des clients')
print('   mca_model.pkl              → Modèle ACM (prince.MCA)')
print('   kmeans_model.pkl           → Modèle K-Means')
print('   mca_metadata.pkl           → Paramètres & résultats')

---
##  Cellule 16 : Synthèse et Conclusions Métier

Cette cellule résume les insights clés de l'analyse.

In [ ]:
# ============================================================
# BLOC 16 — SYNTHÈSE FINALE
# ============================================================

print('=' * 70)
print('SYNTHÈSE DE L\'ANALYSE ACM — RISQUE CRÉDIT BANCAIRE')
print('=' * 70)

print('''
 OBJECTIF
   Identifier les profils de clients à risque à travers une ACM
   sur 6 variables socio-comportementales.

 RÉSULTATS CLÉS

   1. ASSOCIATIONS (V de Cramér)
      • Les variables Marche (compte) et Impaye (historique) sont
        les plus fortement associées au risque.
      • Endettement et Assurance forment un axe secondaire important.

   2. AXES FACTORIELS
      • Axe 1 : Oppose risque FAIBLE (compte positif, aucun impayé,
                endettement faible) vs risque ÉLEVÉ (compte débiteur,
                impayés répétés, aucune assurance)
      • Axe 2 : Différencie par TYPE DE CRÉDIT et PROFESSION
                (immobilier/cadres vs consommation/sans emploi)

   3. SEGMENTS (K-Means sur axes ACM)
      • Segment "Bon" (~40%) : Cadres/fonctionnaires, compte positif,
        bon apport, crédit immobilier → RISQUE FAIBLE
      • Segment "Moyen" (~35%) : Employés qualifiés, compte nul,
        quelques impayés → RISQUE MODÉRÉ
      • Segment "Risqué" (~25%) : Sans emploi/ouvriers, compte débiteur,
        impayés répétés, aucune assurance → RISQUE ÉLEVÉ

 RECOMMANDATIONS MÉTIER
   • Renforcer les critères Marche + Impaye dans le scoring
   • Proposer des assurances obligatoires au Segment 3
   • Adapter les offres crédit par segment (immobilier vs consommation)
   • Surveiller le taux d'endettement combiné avec l'historique

 STACK TECHNIQUE
   Python · pandas · numpy · prince (MCA) · sklearn (K-Means)
   matplotlib · seaborn · plotly · Streamlit (app déployable)
''')
print('=' * 70)
print(' Analyse complète — Prêt pour le déploiement Streamlit !')
print('=' * 70)